# AWD Satellite Analysis Pipeline

This notebook demonstrates the end-to-end pipeline for predicting whether Alternate Wetting and Drying (AWD) practices were followed on farms, using Random Forest. Models are trained on a set of farms and validated on unseen farms to ensure generalizability.

In [ ]:
import sys
import os

# Add the src directory to the path so we can import our modules
sys.path.append(os.path.abspath('../src'))

from preprocessing import load_data, preprocess_data
from feature_engineering import calculate_temporal_features
from model import train_random_forest, evaluate_model
from results import plot_feature_importances, plot_confusion_matrix, print_metrics_report

## 1. Data Loading

In [ ]:
# Load the dataset. This will throw an error if the real dataset is not available.
data_path = '../data/processed/combined_satellite_timeseries.csv'
df = load_data(data_path)

print(f"Data loaded. Shape: {df.shape}")
df.head()

## 2. Feature Engineering

In [ ]:
# Calculate temporal and rolling features
df_engineered = calculate_temporal_features(df)

print(f"Feature engineering complete. Shape: {df_engineered.shape}")
df_engineered.head()

## 3. Preprocessing (Farm-level Group Splitting)

In [ ]:
# Preprocess the data and split by farm to ensure model generalizes to unseen farms.
# Note: Feature scaling is omitted as Random Forest doesn't require it.
X_train, X_test, y_train, y_test = preprocess_data(df_engineered, target_col='awd_followed', group_col='farm_id')

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## 4. Model Training (Random Forest)

In [ ]:
# Exclude non-predictive columns from training (like ID or dates)
cols_to_drop = ['farm_id', 'date']
X_train_features = X_train.drop(columns=[col for col in cols_to_drop if col in X_train.columns])
X_test_features = X_test.drop(columns=[col for col in cols_to_drop if col in X_test.columns])

# Train the Random Forest model
rf_model = train_random_forest(X_train_features, y_train, n_estimators=150, max_depth=10)

print("Model training complete.")

## 5. Evaluation and Results

In [ ]:
# Evaluate the model on the test set (unseen farms)
metrics, y_pred = evaluate_model(rf_model, X_test_features, y_test)

# Print textual report
print_metrics_report(metrics)

In [ ]:
# Plot the confusion matrix
plot_confusion_matrix(y_test, y_pred)

In [ ]:
# Plot feature importances to see which satellite/IoT metrics are most predictive
plot_feature_importances(rf_model, X_train_features.columns)